# Loading measurement data

Single-file parquet exports of the LZR and ZMap TCP anycast measurements,
plus the Hive-partitioned ZGrab results.
All datasets cover **2026-02-04 to 2026-03-03** across three vantage points
(`au-syd`, `de-mun`, `nl-ens`).

| file | rows | on disk | format |
|------|------|---------|--------|
| `lzr.parquet` | 2,052,017,129 | 411 MB | single file |
| `zmap.parquet` | 2,066,766,778 | 417 MB | single file |
| `zgrab_partitioned.tar.xz` | ~2M | 9.6 GB | Hive-partitioned archive |

LZR and ZMap are **too large to load fully into a pandas DataFrame** (~49 GB / ~29 GB).
Use **pyarrow** for filtered/streaming reads or **DuckDB** for SQL queries.

ZGrab is partitioned by `vp/port/date` — individual partitions fit in memory but
the full `data` struct is very wide (~2300 leaf columns), so always select specific
protocol columns.

### Schema — lzr.parquet

| column | type | description |
|--------|------|-------------|
| `saddr` | `uint32` | Source IP (packed; use `u32_to_ip_str` to convert back) |
| `fingerprint` | `string` | Protocol fingerprint (32 distinct: http, tls, ssh, …) |
| `vp` | `dictionary<string>` | Vantage point |
| `port` | `uint16` | Destination port |
| `date` | `date32` | Measurement date |

### Schema — zmap.parquet

| column | type | description |
|--------|------|-------------|
| `saddr` | `uint32` | Source IP (packed; use `u32_to_ip_str` to convert back) |
| `vp` | `dictionary<string>` | Vantage point |
| `port` | `uint16` | Destination port |
| `date` | `date32` | Measurement date |

### Schema — zgrab (partitioned)

| column | type | description |
|--------|------|-------------|
| `ip` | `string` | IPv4 address (dotted-quad) |
| `data` | `struct` | Nested protocol results (~34 top-level protocol fields) |
| `vp` | `string` | Vantage point (from partition) |
| `port` | `string` | Port group: `all_ports`, `ssh_ports`, `email_ports` (from partition) |
| `year`, `month`, `day` | `int32` | Measurement date components (from partition) |

In [1]:
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pyarrow.compute as pc
import pandas as pd
import duckdb

LZR_PATH  = "lzr.parquet"
ZMAP_PATH = "zmap.parquet"


def u32_to_ip_str(u: np.ndarray) -> np.ndarray:
    """Convert a uint32 array to dotted-quad IP strings."""
    u = np.asarray(u, dtype=np.uint32)
    return (
        (u >> 24).astype(str) + "." +
        ((u >> 16) & 0xFF).astype(str) + "." +
        ((u >> 8) & 0xFF).astype(str) + "." +
        (u & 0xFF).astype(str)
    )


def ip_str_to_u32(ips: list[str] | np.ndarray) -> np.ndarray:
    """Convert dotted-quad IP strings to uint32."""
    parts = pd.Series(ips).str.split(".", expand=True).astype(np.uint32)
    return (parts[0].values << 24) | (parts[1].values << 16) | (parts[2].values << 8) | parts[3].values

## Inspect the file metadata (no data loaded)

In [2]:
for path in [LZR_PATH, ZMAP_PATH]:
    pf = pq.ParquetFile(path)
    meta = pf.metadata
    print(f"=== {path} ===")
    print(f"  rows:       {meta.num_rows:>15,}")
    print(f"  row groups: {meta.num_row_groups:>15,}")
    print(f"  columns:    {', '.join(f.name for f in pf.schema_arrow)}")
    print()

=== lzr.parquet ===
  rows:         2,052,017,129
  row groups:           1,968
  columns:    saddr, fingerprint, vp, port, date

=== zmap.parquet ===
  rows:         2,066,766,778
  row groups:           1,982
  columns:    saddr, vp, port, date



## Filtered loading with pyarrow

Use column and row-group filters to read only the subset you need.
This avoids loading 2B rows into memory.

In [3]:
import datetime

# Load a subset: single vantage point, single port, one week
# Filters are pushed down to row groups — only matching groups are read from disk

lzr_subset = (
    ds.dataset(LZR_PATH, format="parquet")
    .to_table(
        filter=(
            (ds.field("vp") == "nl-ens")
            & (ds.field("fingerprint") != "unknown")
            & (ds.field("port") == 443)
            & (ds.field("date") >= datetime.date(2026, 2, 14))
            & (ds.field("date") <= datetime.date(2026, 2, 21))
        ),
    )
    .to_pandas()
)

lzr_subset["saddr"] = u32_to_ip_str(lzr_subset["saddr"].values)
print(f"lzr subset: {len(lzr_subset):,} rows")
lzr_subset.head(10)

lzr subset: 2,648,371 rows


,saddr,fingerprint,vp,port,date
0,1.0.0.3,tls,nl-ens,443,2026-02-14
1,1.0.0.3,tls,nl-ens,443,2026-02-16
2,1.0.0.4,tls,nl-ens,443,2026-02-14
3,1.0.0.6,tls,nl-ens,443,2026-02-18
4,1.0.0.7,tls,nl-ens,443,2026-02-15
5,1.0.0.8,tls,nl-ens,443,2026-02-16
6,1.0.0.9,tls,nl-ens,443,2026-02-17
7,1.0.0.14,tls,nl-ens,443,2026-02-18
8,1.0.0.15,tls,nl-ens,443,2026-02-16
9,1.0.0.16,tls,nl-ens,443,2026-02-14


In [4]:
# Same for zmap
zmap_subset = (
    ds.dataset(ZMAP_PATH, format="parquet")
    .to_table(
        filter=(
            (ds.field("vp") == "nl-ens")
            & (ds.field("port") == 443)
            & (ds.field("date") >= datetime.date(2026, 2, 14))
            & (ds.field("date") <= datetime.date(2026, 2, 21))
        ),
    )
    .to_pandas()
)

zmap_subset["saddr"] = u32_to_ip_str(zmap_subset["saddr"].values)
print(f"zmap subset: {len(zmap_subset):,} rows")
zmap_subset.head(10)

zmap subset: 12,822,086 rows


,saddr,vp,port,date
0,1.0.0.0,nl-ens,443,2026-02-14
1,1.0.0.0,nl-ens,443,2026-02-15
2,1.0.0.0,nl-ens,443,2026-02-16
3,1.0.0.0,nl-ens,443,2026-02-17
4,1.0.0.1,nl-ens,443,2026-02-14
5,1.0.0.1,nl-ens,443,2026-02-15
6,1.0.0.1,nl-ens,443,2026-02-16
7,1.0.0.1,nl-ens,443,2026-02-17
8,1.0.0.2,nl-ens,443,2026-02-14
9,1.0.0.2,nl-ens,443,2026-02-15


In [5]:
# Loading the full parquets is infeasible due to its large size.
# A single vp × date slice is ~24M rows for lzr / ~25M for zmap — fits in memory.
# DuckDB handles non-saddr columns; pyarrow reads saddr (BYTE_STREAM_SPLIT uint32).

def load_day(parquet_path: str, vp: str, date: datetime.date) -> pd.DataFrame:
    """Load all rows for a single vp × date into a DataFrame."""
    # Fast path: non-saddr columns via DuckDB
    df = duckdb.sql(f"""
        SELECT * EXCLUDE (saddr)
        FROM '{parquet_path}'
        WHERE vp = '{vp}' AND date = '{date}'
    """).df()

    # saddr via pyarrow (only way to read BYTE_STREAM_SPLIT uint32)
    saddr = (
        ds.dataset(parquet_path, format="parquet")
        .to_table(
            columns=["saddr"],
            filter=(
                (ds.field("vp") == vp)
                & (ds.field("date") == date)
            ),
        )
        .column("saddr")
        .to_numpy()
    )
    df.insert(0, "saddr", u32_to_ip_str(saddr))
    return df


# Example: load a single day from a single vp (all ports)
lzr_day  = load_day(LZR_PATH,  "nl-ens", datetime.date(2026, 2, 14))
zmap_day = load_day(ZMAP_PATH, "nl-ens", datetime.date(2026, 2, 14))

print(f"lzr  single day: {len(lzr_day):>12,} rows, {lzr_day.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(f"zmap single day: {len(zmap_day):>12,} rows, {zmap_day.memory_usage(deep=True).sum() / 1e6:.0f} MB")
lzr_day.head()

# Free memory
del lzr_day, zmap_day

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

lzr  single day:   87,338,353 rows, 5227 MB
zmap single day:   87,336,978 rows, 3922 MB


## IP lookups and joins

Since the full dataset doesn't fit in memory, use these functions to look up
specific IPs or join a list of IPs against the dataset.

`saddr` is stored as `uint32` — the functions below handle the conversion.

In [6]:
def lookup_ips(parquet_path: str, ips: list[str], extra_filter=None) -> pd.DataFrame:
    """Look up one or more IPs in a parquet file.

    Parameters
    ----------
    parquet_path : str
        Path to lzr.parquet or zmap.parquet.
    ips : list[str]
        Dotted-quad IP addresses to look up.
    extra_filter : pyarrow expression, optional
        Additional filter (e.g. ds.field("port") == 443).

    Returns
    -------
    pd.DataFrame with matching rows, saddr converted back to string.
    """
    u32_vals = ip_str_to_u32(ips)
    ip_filter = ds.field("saddr").isin(u32_vals.tolist())
    if extra_filter is not None:
        ip_filter = ip_filter & extra_filter

    tbl = ds.dataset(parquet_path, format="parquet").to_table(filter=ip_filter)
    df = tbl.to_pandas()
    df["saddr"] = u32_to_ip_str(df["saddr"].values)
    return df


def join_ips(parquet_path: str, ip_df: pd.DataFrame, ip_col: str = "ip",
             extra_filter=None) -> pd.DataFrame:
    """Inner-join a DataFrame of IPs against a parquet dataset.

    Parameters
    ----------
    parquet_path : str
        Path to lzr.parquet or zmap.parquet.
    ip_df : pd.DataFrame
        DataFrame containing an IP column to join on.
    ip_col : str
        Name of the IP column in ip_df (dotted-quad strings).
    extra_filter : pyarrow expression, optional
        Additional filter pushed down before the join.

    Returns
    -------
    pd.DataFrame — inner join of ip_df and the parquet dataset on saddr.
    """
    unique_ips = ip_df[ip_col].unique().tolist()
    matched = lookup_ips(parquet_path, unique_ips, extra_filter=extra_filter)
    return matched.merge(ip_df, left_on="saddr", right_on=ip_col, how="inner")

In [7]:
# Example: look up specific IPs in LZR
results = lookup_ips(LZR_PATH, ["1.0.0.1", "8.8.8.8", "9.9.9.9"])
print(f"Found {len(results):,} rows for 3 IPs")
results.head(10)

Found 181 rows for 3 IPs


,saddr,fingerprint,vp,port,date
0,1.0.0.1,unknown,au-syd,2000,2026-02-14
1,1.0.0.1,unknown,au-syd,2000,2026-02-15
2,1.0.0.1,unknown,au-syd,2000,2026-02-16
3,1.0.0.1,unknown,au-syd,2000,2026-02-17
4,1.0.0.1,unknown,au-syd,2000,2026-02-18
5,1.0.0.1,unknown,au-syd,2000,2026-03-03
6,8.8.8.8,unknown,au-syd,2000,2026-02-14
7,8.8.8.8,unknown,au-syd,2000,2026-02-15
8,8.8.8.8,unknown,au-syd,2000,2026-02-16
9,8.8.8.8,unknown,au-syd,2000,2026-02-17


In [8]:
# Example: look up the same IPs in ZMap, filtered to port 443
results_zmap = lookup_ips(ZMAP_PATH, ["1.0.0.1", "8.8.8.8", "9.9.9.9"],
                          extra_filter=ds.field("port") == 443)
print(f"Found {len(results_zmap):,} rows")
results_zmap.head(10)

Found 69 rows


,saddr,vp,port,date
0,1.0.0.1,au-syd,443,2026-02-14
1,1.0.0.1,au-syd,443,2026-02-15
2,1.0.0.1,au-syd,443,2026-02-16
3,1.0.0.1,au-syd,443,2026-02-17
4,1.0.0.1,au-syd,443,2026-02-18
5,1.0.0.1,au-syd,443,2026-03-03
6,8.8.8.8,au-syd,443,2026-02-14
7,8.8.8.8,au-syd,443,2026-02-15
8,8.8.8.8,au-syd,443,2026-02-16
9,8.8.8.8,au-syd,443,2026-02-17


In [9]:
# Example: join an external DataFrame of IPs against ZMap
my_ips = pd.DataFrame({
    "ip": ["1.0.0.1", "8.8.8.8", "9.9.9.9"],
    "label": ["Cloudflare", "Google DNS", "Quad9"],
})

joined = join_ips(ZMAP_PATH, my_ips, ip_col="ip",
                  extra_filter=ds.field("port") == 443)
print(f"Joined: {len(joined):,} rows")
joined.head(10)

Joined: 69 rows


,saddr,vp,port,date,ip,label
0,1.0.0.1,au-syd,443,2026-02-14,1.0.0.1,Cloudflare
1,1.0.0.1,au-syd,443,2026-02-15,1.0.0.1,Cloudflare
2,1.0.0.1,au-syd,443,2026-02-16,1.0.0.1,Cloudflare
3,1.0.0.1,au-syd,443,2026-02-17,1.0.0.1,Cloudflare
4,1.0.0.1,au-syd,443,2026-02-18,1.0.0.1,Cloudflare
5,1.0.0.1,au-syd,443,2026-03-03,1.0.0.1,Cloudflare
6,8.8.8.8,au-syd,443,2026-02-14,8.8.8.8,Google DNS
7,8.8.8.8,au-syd,443,2026-02-15,8.8.8.8,Google DNS
8,8.8.8.8,au-syd,443,2026-02-16,8.8.8.8,Google DNS
9,8.8.8.8,au-syd,443,2026-02-17,8.8.8.8,Google DNS


## DuckDB for SQL queries

DuckDB can query parquet files directly with no loading step. Much faster than
streaming batches in Python for aggregations.

> **Note:** DuckDB cannot read the `saddr` column (it uses `BYTE_STREAM_SPLIT`
> encoding on `uint32`, which DuckDB only supports for float/double). Use pyarrow
> for any query that touches `saddr`, or use the `lookup_ips`/`join_ips` functions above.

In [10]:
import duckdb

# Row counts per vp and date
duckdb.sql("""
    SELECT vp, date, count(*) AS rows
    FROM 'lzr.parquet'
    GROUP BY vp, date
    ORDER BY vp, date
    LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,vp,date,rows
0,au-syd,2026-02-14,96584608
1,au-syd,2026-02-15,96515376
2,au-syd,2026-02-16,96358071
3,au-syd,2026-02-17,96268103
4,au-syd,2026-02-18,96176110
5,au-syd,2026-03-03,94949130
6,de-mun,2026-03-03,87816780
7,nl-ens,2026-02-04,87145922
8,nl-ens,2026-02-05,86623122
9,nl-ens,2026-02-06,86668410


In [11]:
# Top fingerprints by frequency (lzr only)
duckdb.sql("""
    SELECT fingerprint, count(*) AS n,
           round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
    FROM 'lzr.parquet'
    GROUP BY fingerprint
    ORDER BY n DESC
""").df()

,fingerprint,n,pct
0,unknown,2017800988,98.33
1,tls,18666826,0.91
2,http,14797194,0.72
3,ssh,200172,0.01
4,smtp,169466,0.01
5,imap,94149,0.00
6,pop3,92239,0.00
7,ftp,75978,0.00
8,postgres,28230,0.00
9,rdp,24145,0.00


In [12]:
# Responsive ports per vantage point (zmap)
duckdb.sql("""
    SELECT vp, count(*) AS total_rows,
           count(DISTINCT port) AS ports,
           min(date) AS first_date,
           max(date) AS last_date
    FROM 'zmap.parquet'
    GROUP BY vp
    ORDER BY vp
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,vp,total_rows,ports,first_date,last_date
0,au-syd,591851848,116,2026-02-14,2026-03-03
1,de-mun,87774121,117,2026-03-03,2026-03-03
2,nl-ens,1387140809,117,2026-02-04,2026-03-03


## ZGrab dataset

ZGrab results are distributed as a Hive-partitioned parquet archive
(`zgrab_partitioned.tar.xz`, 9.6 GB compressed, ~10.7 GB extracted).

The partition structure is `format=parquet/vp=.../port=.../year=.../month=.../day=.../`.
Each parquet file contains two physical columns:
- **`ip`** — IPv4 address as string
- **`data`** — deeply nested struct with protocol-specific grab results (2300+ leaf columns)

Partition columns (`vp`, `port`, `year`, `month`, `day`) are inferred from the
directory names via Hive partitioning.

| partition column | values |
|-----------------|--------|
| `vp` | `au-syd`, `de-mun`, `nl-ens` |
| `port` | `all_ports`, `ssh_ports`, `email_ports` |

In [13]:
import subprocess
from pathlib import Path

ZGRAB_ARCHIVE = Path("zgrab_partitioned.tar.xz")
ZGRAB_DIR     = Path("zgrab_partitioned")   # extraction target
ZGRAB_PATH    = ZGRAB_DIR / "format=parquet" # pyarrow dataset root

if not ZGRAB_PATH.exists():
    print(f"Extracting {ZGRAB_ARCHIVE} → {ZGRAB_DIR}/ ...")
    ZGRAB_DIR.mkdir(exist_ok=True)
    # Use system tar for xz — much faster than Python's tarfile module
    subprocess.run(
        ["tar", "-xJf", str(ZGRAB_ARCHIVE), "-C", str(ZGRAB_DIR)],
        check=True,
    )
    print("Done.")
else:
    print(f"Already extracted at {ZGRAB_PATH}")

Already extracted at zgrab_partitioned/format=parquet


In [14]:
# Open the dataset with Hive partitioning (vp, port, year, month, day from dirs)
zgrab = ds.dataset(ZGRAB_PATH, format="parquet", partitioning="hive")

# Basic stats
fragments = list(zgrab.get_fragments())
total_rows = sum(f.metadata.num_rows for f in fragments)
print(f"zgrab: {len(fragments)} partition files, {total_rows:,} total rows")
print(f"schema ({len(zgrab.schema)} columns):")
for f in zgrab.schema:
    if f.name != "data":
        print(f"  {f.name}: {f.type}")
print(f"  data: struct<...> ({len(zgrab.schema.field('data').type)} top-level fields)")

zgrab: 10 partition files, 4,819,832 total rows
schema (7 columns):
  ip: string
  vp: string
  port: string
  year: int32
  month: int32
  day: int32
  data: struct<...> (34 top-level fields)


In [15]:
# List the top-level protocol fields inside the `data` struct
data_type = zgrab.schema.field("data").type
for i in range(len(data_type)):
    f = data_type.field(i)
    print(f"  data.{f.name}: {f.type.num_fields if hasattr(f.type, 'num_fields') else ''} ({f.type})"[:100])

  data.amqp: 6 (struct<error: string, port: int64, protocol: string, result: struct<auth_success: bo
  data.amqps: 6 (struct<error: string, port: int64, protocol: string, result: struct<auth_success: b
  data.bacnet: 5 (struct<error: string, port: int64, protocol: string, status: string, timestamp: st
  data.banner: 6 (struct<error: string, port: int64, protocol: string, result: struct<banner: string
  data.dnp3: 5 (struct<error: string, port: int64, protocol: string, status: string, timestamp: stri
  data.fox: 6 (struct<error: string, port: int64, protocol: string, result: string, status: string, 
  data.ftp: 6 (struct<error: string, port: int64, protocol: string, result: struct<banner: string>, 
  data.ftps: 6 (struct<error: string, port: int64, protocol: string, result: struct<banner: string, 
  data.http: 6 (struct<error: string, port: int64, protocol: string, result: struct<response: struct
  data.https: 6 (struct<error: string, port: int64, protocol: string, result: struct<respon

### Loading a zgrab partition

The `data` struct is very wide (~2300 leaf columns) so always select specific
columns rather than reading everything.

For nested fields, use a **projection dict** to extract sub-structs:
```python
columns={"http": ds.field("data", "http"), ...}
```
The helper functions below also accept dot notation (`"data.http"`) which is
converted automatically.

In [16]:
# Load a single partition: just ip + partition columns (no data struct)
# This is fast — reads only the lightweight columns
zgrab_ips = (
    zgrab.to_table(
        columns=["ip", "vp", "port", "year", "month", "day"],
        filter=(
            (ds.field("vp") == "nl-ens")
            & (ds.field("port") == "all_ports")
            & (ds.field("year") == 2026)
            & (ds.field("month") == 2)
            & (ds.field("day") == 20)
        ),
    )
    .to_pandas()
)
print(f"zgrab IPs for nl-ens/all_ports/2026-02-20: {len(zgrab_ips):,} rows")
zgrab_ips.head()

zgrab IPs for nl-ens/all_ports/2026-02-20: 761,257 rows


,ip,vp,port,year,month,day
0,198.29.25.139,nl-ens,all_ports,2026,2,20
1,134.224.208.240,nl-ens,all_ports,2026,2,20
2,34.152.90.164,nl-ens,all_ports,2026,2,20
3,34.153.4.47,nl-ens,all_ports,2026,2,20
4,172.67.67.122,nl-ens,all_ports,2026,2,20


In [17]:
# Load a partition with specific protocol data (e.g. HTTP responses)
# Use projection dict to select nested fields from the data struct
zgrab_http = (
    zgrab.to_table(
        columns={
            "ip": ds.field("ip"),
            "http": ds.field("data", "http"),
            "vp": ds.field("vp"),
            "port": ds.field("port"),
        },
        filter=(
            (ds.field("vp") == "nl-ens")
            & (ds.field("port") == "all_ports")
            & (ds.field("year") == 2026)
            & (ds.field("month") == 2)
            & (ds.field("day") == 20)
        ),
    )
    .to_pandas()
)
print(f"zgrab HTTP data: {len(zgrab_http):,} rows, {zgrab_http.memory_usage(deep=True).sum() / 1e6:.0f} MB")
zgrab_http.head()

zgrab HTTP data: 761,257 rows, 58 MB


,ip,http,vp,port
0,198.29.25.139,None,nl-ens,all_ports
1,134.224.208.240,None,nl-ens,all_ports
2,34.152.90.164,None,nl-ens,all_ports
3,34.153.4.47,None,nl-ens,all_ports
4,172.67.67.122,None,nl-ens,all_ports


In [18]:
# Look up specific IPs in zgrab
def _resolve_columns(columns: list[str]) -> dict:
    """Convert a list of column names (with optional dot notation) to a
    pyarrow projection dict.  'data.http' → ds.field("data", "http")
    aliased as 'http' in the output DataFrame."""
    projection = {}
    for col in columns:
        if "." in col:
            parts = col.split(".", maxsplit=1)
            projection[parts[1]] = ds.field(*parts)
        else:
            projection[col] = ds.field(col)
    return projection


def zgrab_lookup_ips(ips: list[str], columns: list[str] | None = None,
                     extra_filter=None) -> pd.DataFrame:
    """Look up IPs in the zgrab dataset.

    Parameters
    ----------
    ips : list[str]
        Dotted-quad IP addresses to look up.
    columns : list[str], optional
        Columns to read. Defaults to ip + partition columns (no data struct).
        Use dot notation for nested fields, e.g. ["ip", "data.http", "vp", "port"].
        Nested fields are aliased without the 'data.' prefix in the output.
    extra_filter : pyarrow expression, optional
        Additional filter (e.g. ds.field("port") == "all_ports").
    """
    if columns is None:
        columns = ["ip", "vp", "port", "year", "month", "day"]

    ip_filter = ds.field("ip").isin(ips)
    if extra_filter is not None:
        ip_filter = ip_filter & extra_filter

    projection = _resolve_columns(columns)
    return zgrab.to_table(columns=projection, filter=ip_filter).to_pandas()


# Example: look up IPs across all partitions (lightweight — no data struct)
zgrab_results = zgrab_lookup_ips(["1.0.0.1", "8.8.8.8", "9.9.9.9"])
print(f"Found {len(zgrab_results):,} rows")
zgrab_results.head(10)

Found 10 rows


,ip,vp,port,year,month,day
0,9.9.9.9,au-syd,all_ports,2026,2,20
1,8.8.8.8,au-syd,all_ports,2026,2,20
2,9.9.9.9,au-syd,all_ports,2026,3,4
3,8.8.8.8,au-syd,all_ports,2026,3,4
4,9.9.9.9,de-mun,all_ports,2026,3,4
5,8.8.8.8,de-mun,all_ports,2026,3,4
6,9.9.9.9,nl-ens,all_ports,2026,2,20
7,8.8.8.8,nl-ens,all_ports,2026,2,20
8,9.9.9.9,nl-ens,all_ports,2026,3,4
9,8.8.8.8,nl-ens,all_ports,2026,3,4


In [19]:
# Look up IPs with protocol data — scope to a single partition to avoid OOM
# Available protocols: amqp, amqps, bacnet, banner, dnp3, fox, ftp, ftps,
# http, https, imap, imaps, ipp, managesieve, memcached, modbus, mongodb,
# mqtt, mssql, mysql, oracle, pop3, pop3s, postgres, pptp, redis, siemens,
# smb, smtp, smtps, smtpss, socks5, ssh, telnet
zgrab_results_http = zgrab_lookup_ips(
    ["1.0.0.1", "8.8.8.8"],
    columns=["ip", "data.http", "data.ssh", "vp", "port"],
    extra_filter=(
        (ds.field("port") == "all_ports")
        & (ds.field("vp") == "nl-ens")
        & (ds.field("year") == 2026)
        & (ds.field("month") == 2)
        & (ds.field("day") == 20)
    ),
)
print(f"Found {len(zgrab_results_http):,} rows")
zgrab_results_http.head()

Found 1 rows


,ip,http,ssh,vp,port
0,8.8.8.8,None,None,nl-ens,all_ports
